# U07 练习 | GRU 原理与公式推导

**本练习目标**：
- 验证 GRU 参数量与理论推导一致
- 手写 GRUCell 复现公式，加深公式理解
- 用 nn.GRU 完成序列分类任务
- 通过默写关卡检验掌握程度

**通关条件**：7.1 ~ 7.4 全部通过

## 练习 7.1：验证 GRU 参数量

**背景**：GRU 有 3 组参数（重置门、更新门、候选状态），每组包含 W_x、W_h、bias。

**要求**：
1. 创建 `input_size=32, hidden_size=64` 的 `nn.GRU` 和 `nn.RNN`
2. 分别统计参数量
3. 手动计算 GRU 理论参数量并验证一致
4. 打印 GRU 的所有参数名称和形状，确认 3 组参数结构

**验证**：`gru_params == 3 * rnn_params` 应为 True（近似）

In [ ]:
import torch
import torch.nn as nn

input_size = 32
hidden_size = 64

rnn = nn.RNN(input_size, hidden_size, batch_first=True)
gru = nn.GRU(input_size, hidden_size, batch_first=True)

# TODO 1: 统计 rnn 和 gru 的总参数量
rnn_params = None   # 提示：sum(p.numel() for p in model.parameters())
gru_params = None
print('RNN 参数量:', rnn_params)
print('GRU 参数量:', gru_params)
print('比值:', gru_params / rnn_params)

# TODO 2: 手动计算 GRU 理论参数量
# 每组参数：W_x(input*hidden) + W_h(hidden*hidden) + b_x(hidden) + b_h(hidden)
# GRU 共 3 组
gru_manual = None   # 填写你的计算表达式
print('\nGRU 手算:', gru_manual)
print('与实际一致:', gru_manual == gru_params)

# TODO 3: 打印 GRU 所有参数名和形状，观察 weight_ih/weight_hh 的命名
for name, param in gru.named_parameters():
    print(f'{name}: {param.shape}')

## 练习 7.2：手写 GRUCell，复现公式

**背景**：`nn.GRU` 内部每个时间步做一次 GRUCell 计算。自己实现 GRUCell，能帮助真正理解 4 个公式。

**要求**：
1. 实现 `MyGRUCell`，按照 4 个公式逐步计算
2. 与 `nn.GRUCell` 输出对比（使用相同权重），结果应完全一致

**4 个公式**：
$$r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)$$
$$z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z)$$
$$\tilde{h}_t = \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h)$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MyGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # 重置门参数
        self.W_r = nn.Linear(input_size, hidden_size)
        self.U_r = nn.Linear(hidden_size, hidden_size, bias=False)

        # TODO 1: 定义更新门参数 W_z, U_z（参考重置门）

        # TODO 2: 定义候选状态参数 W_h, U_h（参考重置门）

    def forward(self, x, h_prev):
        # x: (batch, input_size)
        # h_prev: (batch, hidden_size)

        # TODO 3: 计算重置门
        # r = sigmoid(W_r(x) + U_r(h_prev))
        r = None

        # TODO 4: 计算更新门
        # z = sigmoid(W_z(x) + U_z(h_prev))
        z = None

        # TODO 5: 计算候选隐状态
        # h_tilde = tanh(W_h(x) + U_h(r * h_prev))
        h_tilde = None

        # TODO 6: 计算最终隐状态（线性插值）
        # h_new = (1 - z) * h_prev + z * h_tilde
        h_new = None

        return h_new


# 验证：与 nn.GRUCell 输出对比
torch.manual_seed(42)
batch, input_size, hidden_size = 4, 8, 16

x = torch.randn(batch, input_size)
h = torch.randn(batch, hidden_size)

# 官方 GRUCell
official = nn.GRUCell(input_size, hidden_size)
h_official = official(x, h)

# 你的实现（注意：需要从 official 复制权重才能对比）
my_cell = MyGRUCell(input_size, hidden_size)

# 运行你的 forward
h_mine = my_cell(x, h)

print('my_cell 输出 shape:', h_mine.shape if h_mine is not None else 'None')
print('通关条件：shape == (4, 16)')

## 练习 7.3：用 GRU 完成序列分类

**背景**：复用 U06 的数据生成逻辑（词 0-499 偏向类别 0，词 500-999 偏向类别 1），把模型从 RNN 换成 GRU。

**要求**：
1. 实现 `GRUClassifier`（参考 U06 的 `SeqClassifier`，只改 RNN → GRU）
2. 训练 10 个 epoch，验证集准确率应 > 85%
3. 打印每个 epoch 的 train_loss 和 val_acc

**通关标准**：最终 val_acc > 85%

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ===== 数据生成（与 U06 相同）=====
VOCAB_SIZE = 1000
EMBED_DIM = 32
HIDDEN_DIM = 64

def make_sentences(n):
    sentences, labels = [], []
    for _ in range(n):
        seq_len = torch.randint(5, 11, (1,)).item()
        bias = torch.randint(0, 2, (1,)).item()
        if bias == 0:
            indices = torch.randint(0, 500, (seq_len,))
        else:
            indices = torch.randint(500, 1000, (seq_len,))
        label = int((indices[:3] >= 500).sum().item() >= 2)
        sentences.append(indices)
        labels.append(torch.tensor(label, dtype=torch.long))
    return sentences, torch.stack(labels)

def collate_fn(batch):
    sentences, labels = zip(*batch)
    seq_len = max(s.size(0) for s in sentences)
    padded = torch.zeros(len(sentences), seq_len, dtype=torch.long)
    for i, s in enumerate(sentences):
        padded[i, :s.size(0)] = s
    return padded, torch.stack(labels)

class SimpleDataset(Dataset):
    def __init__(self, sentences, labels):
        self.sentences, self.labels = sentences, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.sentences[idx], self.labels[idx]

torch.manual_seed(42)
sentences_train, labels_train = make_sentences(800)
sentences_val,   labels_val   = make_sentences(200)

train_loader = DataLoader(SimpleDataset(sentences_train, labels_train),
                          batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(SimpleDataset(sentences_val, labels_val),
                          batch_size=32, shuffle=False, collate_fn=collate_fn)

# ===== TODO: 定义 GRUClassifier =====
class GRUClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: 三个层：Embedding, GRU, Linear
        pass

    def forward(self, x):
        # x: (batch, seq_len)
        # TODO: 前向传播，返回 logits (batch, 2)
        pass


# ===== TODO: 训练循环 =====
model = GRUClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(10):
    # TODO: 训练一个 epoch，计算 train_loss

    # TODO: 验证，计算 val_acc

    pass

print('通关条件：val_acc > 85%')

## 练习 7.4：默写关卡

**不看笔记，把 4 个 GRU 公式填写完整，并用一句话解释每个公式的作用。**

完成后与 lesson.ipynb 第 7 节对照。

**通关标准**：4 个公式全部正确，每个公式能说清楚它在干什么。

In [ ]:
# 默写练习——用注释写出公式和解释，不需要实际运行

# 公式 1：重置门
# r_t = ???
# 作用：

# 公式 2：更新门
# z_t = ???
# 作用：

# 公式 3：候选隐状态
# h_tilde_t = ???
# 作用：

# 公式 4：新隐藏状态
# h_t = ???
# 作用：

# 额外问题（口头回答）：
# Q1: GRU 如何解决 RNN 的梯度消失问题？
# Q2: 重置门和更新门各控制什么？有什么区别？
# Q3: 公式 4 中 (1-z_t) 和 z_t 加起来等于几？为什么这样设计？

print('默写完成后，和 lesson.ipynb 第 7 节对照检查')